# ⛓️ Module 04: LCEL — LangChain Expression Language

---

## What is LCEL?

**LCEL (LangChain Expression Language)** is the core composability layer of LangChain. It uses the **`|` pipe operator** to chain components together — inspired by Unix pipes.

```python
# Every chain is just piped components!
chain = prompt | llm | parser
result = chain.invoke(input)
```

### Why LCEL?

| Feature | Benefit |
|---------|----------|
| **Streaming** | Built-in streaming for every chain |
| **Async** | Native async support |
| **Parallel** | Run branches simultaneously |
| **Batch** | Process multiple inputs at once |
| **Tracing** | Automatic LangSmith integration |
| **Fallbacks** | Add fallback chains on error |
| **Type safety** | Input/output schemas at every step |

---

## The Runnable Interface

Everything in LCEL implements `Runnable`:

```
Runnable
├── .invoke(input)     → single output
├── .stream(input)     → generator of chunks
├── .batch(inputs)     → list of outputs
├── .ainvoke(input)    → async single output
├── .astream(input)    → async generator
└── .abatch(inputs)    → async list of outputs
```

---

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv(dotenv_path="../.env")

from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.3)
print("Setup complete ✅")

## 1️⃣ Basic Chain Composition

In [ ]:
# ============================================================
# The simplest chain: prompt | llm | parser
# ============================================================
prompt = ChatPromptTemplate.from_template(
    "Explain {concept} in exactly 2 sentences."
)
parser = StrOutputParser()

# Compose with pipe operator
chain = prompt | llm | parser

# The chain is a Runnable — call it with invoke
result = chain.invoke({"concept": "neural networks"})
print("Type:", type(result).__name__)
print("Result:", result)

In [ ]:
# ============================================================
# Inspect chain input/output schema
# ============================================================
print("Input schema:")
print(chain.input_schema.model_json_schema())

print("\nOutput schema:")
print(chain.output_schema.model_json_schema())

In [ ]:
# ============================================================
# Streaming through the chain
# ============================================================
print("Streaming output:")
print("-" * 50)

for chunk in chain.stream({"concept": "quantum computing"}):
    print(chunk, end="", flush=True)

print("\n" + "-" * 50)

In [ ]:
# ============================================================
# Batch processing
# ============================================================
concepts = [
    {"concept": "recursion"},
    {"concept": "REST API"},
    {"concept": "Docker containers"},
]

results = chain.batch(concepts)

for concept, result in zip(concepts, results):
    print(f"\n📘 {concept['concept'].upper()}:")
    print(f"   {result}")

## 2️⃣ RunnableLambda — Wrap Any Function

In [ ]:
from langchain_core.runnables import RunnableLambda

# ============================================================
# Turn any Python function into a Runnable
# ============================================================

def word_counter(text: str) -> dict:
    """Count words and characters in text"""
    words = text.split()
    return {
        "text": text,
        "word_count": len(words),
        "char_count": len(text),
        "avg_word_length": sum(len(w) for w in words) / len(words) if words else 0
    }

def format_stats(stats: dict) -> str:
    """Format stats into a readable string"""
    return f"""
Text: "{stats['text'][:50]}..."
Words: {stats['word_count']}
Characters: {stats['char_count']}
Avg word length: {stats['avg_word_length']:.1f}
"""

# Chain: prompt -> llm -> StrOutputParser -> word_counter -> format_stats
analysis_chain = (
    ChatPromptTemplate.from_template("Write a paragraph about {topic}")
    | llm
    | StrOutputParser()
    | RunnableLambda(word_counter)
    | RunnableLambda(format_stats)
)

result = analysis_chain.invoke({"topic": "black holes"})
print(result)

In [ ]:
# ============================================================
# Lambda shorthand — Use Python lambdas directly!
# ============================================================
chain = (
    ChatPromptTemplate.from_template("List 5 {category} in a comma-separated list, nothing else.")
    | llm
    | StrOutputParser()
    | (lambda text: [item.strip() for item in text.split(',')])  # Direct lambda!
    | (lambda items: {"items": items, "count": len(items)})
)

result = chain.invoke({"category": "programming languages"})
print("Result:", result)

## 3️⃣ RunnableParallel — Run Branches Simultaneously

In [ ]:
from langchain_core.runnables import RunnableParallel

# ============================================================
# Run multiple chains at the same time
# ============================================================

# Three different analysis chains
summary_chain = (
    ChatPromptTemplate.from_template("Summarize this topic in 1 sentence: {topic}")
    | llm | StrOutputParser()
)

pros_chain = (
    ChatPromptTemplate.from_template("List 3 pros of {topic} in bullet points")
    | llm | StrOutputParser()
)

cons_chain = (
    ChatPromptTemplate.from_template("List 3 cons of {topic} in bullet points")
    | llm | StrOutputParser()
)

# RunnableParallel runs all three SIMULTANEOUSLY
parallel_analysis = RunnableParallel(
    summary=summary_chain,
    pros=pros_chain,
    cons=cons_chain
)

import time
start = time.time()
result = parallel_analysis.invoke({"topic": "TypeScript vs JavaScript"})
elapsed = time.time() - start

print(f"Completed in {elapsed:.2f}s (would take ~{elapsed*3:.0f}s sequentially)")
print()
print("📋 SUMMARY:")
print(result['summary'])
print("\n✅ PROS:")
print(result['pros'])
print("\n❌ CONS:")
print(result['cons'])

In [ ]:
# ============================================================
# Parallel with passthrough — Keep original input
# ============================================================
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

chain_with_passthrough = RunnableParallel(
    original=RunnablePassthrough(),   # Pass input through unchanged
    improved=(
        ChatPromptTemplate.from_template("Improve this text: {text}")
        | llm | StrOutputParser()
    )
)

result = chain_with_passthrough.invoke({"text": "the cat sat on mat"})
print("Original:", result['original'])
print("Improved:", result['improved'])

## 4️⃣ RunnablePassthrough — Pass Input Unchanged

In [ ]:
from langchain_core.runnables import RunnablePassthrough

# ============================================================
# Classic RAG pattern using RunnablePassthrough
# ============================================================
# Pattern: We need both the original question AND the retrieved context

def fake_retriever(query: str) -> str:
    """Simulates retrieving relevant documents"""
    return "LangChain was created by Harrison Chase in 2022. It is open source."

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer the question using ONLY the provided context."),
    ("human", "Context: {context}\n\nQuestion: {question}")
])

rag_chain = (
    RunnableParallel(
        context=RunnableLambda(fake_retriever),  # Get context
        question=RunnablePassthrough()           # Pass question through
    )
    | rag_prompt
    | llm
    | StrOutputParser()
)

answer = rag_chain.invoke("Who created LangChain?")
print("Answer:", answer)

## 5️⃣ RunnableBranch — Conditional Logic

In [ ]:
from langchain_core.runnables import RunnableBranch

# ============================================================
# Route to different chains based on input
# ============================================================

# Different response chains
technical_chain = (
    ChatPromptTemplate.from_template("Provide a technical, detailed answer to: {question}")
    | llm | StrOutputParser()
)

simple_chain = (
    ChatPromptTemplate.from_template("Explain simply for a beginner: {question}")
    | llm | StrOutputParser()
)

default_chain = (
    ChatPromptTemplate.from_template("Answer this general question: {question}")
    | llm | StrOutputParser()
)

# Branch based on question content
branch = RunnableBranch(
    # (condition, chain_to_use)
    (lambda x: "code" in x["question"].lower() or "algorithm" in x["question"].lower(), technical_chain),
    (lambda x: "simple" in x["question"].lower() or "beginner" in x["question"].lower(), simple_chain),
    default_chain  # Default fallback
)

# Test with different questions
questions = [
    {"question": "How does the quicksort algorithm work?"},
    {"question": "Simple explanation of what Python is for a beginner?"},
    {"question": "What is the history of programming?"}
]

for q in questions:
    print(f"Question: {q['question']}")
    print(f"Answer: {branch.invoke(q)[:100]}...\n")

## 6️⃣ Chain Fallbacks — Handle Failures Gracefully

In [ ]:
from langchain_groq import ChatGroq
from langchain_anthropic import ChatAnthropic

# ============================================================
# Primary chain uses GPT-4, fallback uses Claude
# ============================================================
primary_llm = ChatGroq(model="llama-3.3-70b-versatile")         # More expensive/powerful
fallback_llm = ChatGroq(model="llama-3.1-8b-instant")   # Cheaper fallback

prompt = ChatPromptTemplate.from_template("Answer: {question}")
parser = StrOutputParser()

primary_chain = prompt | primary_llm | parser
fallback_chain = prompt | fallback_llm | parser

# Chain with fallback
robust_chain = primary_chain.with_fallbacks([fallback_chain])

# This will try primary first, then fallback on error
result = robust_chain.invoke({"question": "What is 42?"})
print("Result:", result)

## 7️⃣ Chaining Chains — Sequential Multi-Step Workflows

In [ ]:
# ============================================================
# Multi-step content pipeline
# Step 1: Generate an outline
# Step 2: Write a blog post from that outline
# Step 3: Create a catchy title for the post
# ============================================================

# Step 1: Outline chain
outline_chain = (
    ChatPromptTemplate.from_template("Create a 5-point outline for a blog post about {topic}")
    | llm | StrOutputParser()
)

# Step 2: Blog post chain (takes outline as input)
blog_chain = (
    ChatPromptTemplate.from_template(
        "Write a 200-word blog post based on this outline:\n{outline}"
    )
    | llm | StrOutputParser()
)

# Step 3: Title chain (takes blog post as input)
title_chain = (
    ChatPromptTemplate.from_template(
        "Generate 3 catchy SEO-friendly titles for this blog post:\n{blog_post}"
    )
    | llm | StrOutputParser()
)

# Connect them: the output of each becomes the input of the next
full_pipeline = (
    {"topic": RunnablePassthrough()}   # Input
    | RunnableParallel(
        outline=outline_chain,
        topic=RunnablePassthrough()    # Keep topic for reference
    )
    | {
        "blog_post": ({"outline": lambda x: x["outline"]} | blog_chain),
        "outline": lambda x: x["outline"]
    }
    | RunnableParallel(
        blog_post=lambda x: x["blog_post"],
        titles=({"blog_post": lambda x: x["blog_post"]} | title_chain)
    )
)

result = full_pipeline.invoke({"topic": "the future of AI in healthcare"})

print("✍️ BLOG POST:")
print(result["blog_post"])
print("\n📌 TITLE OPTIONS:")
print(result["titles"])

## 8️⃣ Inspecting Chains with astream_events

In [ ]:
# ============================================================
# astream_events — See what's happening at each step
# ============================================================

chain = (
    ChatPromptTemplate.from_template("Write a one-sentence fact about {topic}")
    | llm
    | StrOutputParser()
)

print("Events from chain execution:")
print("-" * 50)

async for event in chain.astream_events({"topic": "Mars"}, version="v2"):
    kind = event["event"]
    name = event.get("name", "")
    
    if kind == "on_chain_start":
        print(f"🟢 START: {name}")
    elif kind == "on_chain_end":
        print(f"🔴 END: {name}")
    elif kind == "on_chat_model_stream":
        chunk = event["data"]["chunk"].content
        if chunk:
            print(chunk, end="", flush=True)

print("\n" + "-" * 50)

## ✅ Module 04 Summary

You've learned:
- ✅ LCEL pipe `|` operator for chain composition
- ✅ The Runnable interface (invoke, stream, batch, async)
- ✅ `RunnableLambda` — wrap any Python function
- ✅ `RunnableParallel` — run branches simultaneously
- ✅ `RunnablePassthrough` — pass input unchanged
- ✅ `RunnableBranch` — conditional routing
- ✅ Fallbacks for error resilience
- ✅ Multi-step sequential workflows

### 🚀 Next: [Module 05 — Memory & Conversation History](05_Memory_and_Conversations.ipynb)